# 02 · Transfer Learning Model & Preprocessing
**Chest X-Ray Classification with ResNet50 Transfer Learning (TensorFlow / Keras)**  
*Task 2 · Sections 2.1-2.6* · MSc Data Science group project

**In this notebook**
- Why ResNet50: ImageNet pretraining and residual connections
- CLAHE contrast enhancement and medically safe augmentation (no horizontal flips)
- 15% stratified validation subset (948 images) and balanced class weights
- Custom 512 → 256 → 128 → 4 head with batch normalisation, dropout and label smoothing

> **How to use these notebooks.** The outputs below come from one complete run on a Kaggle Tesla T4 GPU. Each part builds on objects created earlier (data generators, the trained model, predictions), so the part notebooks are for reading and review. To reproduce the results, run [`00_full_pipeline_run_all.ipynb`](00_full_pipeline_run_all.ipynb) top to bottom.

[← Dataset Understanding & Exploration](01_dataset_exploration.ipynb) · [Three-Phase Fine-Tuning →](03_three_phase_fine_tuning.ipynb)

---

## Task 2 — Transfer Learning Model Implementation

This section covers selecting and justifying ResNet50 as the pre-trained backbone, building the transfer learning pipeline, and training the model using a 3-phase fine-tuning strategy with techniques to handle overfitting.

### 2.1 Model Selection & Justification

**Why ResNet50?**

ResNet50 (Residual Network, 50 layers) was originally trained on ImageNet (1.2M images, 1000 classes) for general object recognition. Key reasons for selecting it for chest X-ray classification:

| Property | Detail |
|---|---|
| Architecture | 50-layer deep residual network with skip connections |
| Pre-trained weights | ImageNet — learns general low-level features (edges, textures) transferable to X-rays |
| Input size | 224×224×3 — standard medical image resolution |
| Skip connections | Prevent vanishing gradients — critical for deep fine-tuning |
| Widely used in medical imaging | Residual CNNs are a standard, well-studied backbone for chest X-ray classification (the landmark CheXNet model used the closely related DenseNet-121) |

The classification head is replaced with a custom deeper head: **512 → 256 → 128 → 4 classes**, with BatchNormalization and Dropout at each layer.

### 2.2 Import Deep Learning Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, GlobalAveragePooling2D,
                                      Dropout, BatchNormalization)
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                                         ModelCheckpoint, LearningRateScheduler)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from sklearn.utils.class_weight import compute_class_weight
import cv2

print("✅ TensorFlow version:", tf.__version__)
print("✅ All libraries imported")


2026-03-17 04:42:18.546455: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773722538.728129      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773722538.787018      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773722539.245529      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773722539.245578      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773722539.245581      55 computation_placer.cc:177] computation placer alr

✅ TensorFlow version: 2.19.0
✅ All libraries imported


### 2.3 Hyperparameter Configuration

Key configuration decisions:
- **Validation split of 15%** (~948 images) replaces the original 38-image val folder, which caused extremely noisy/jumpy validation graphs
- **Phase 2 extended to 35 epochs** to give more time for the deeper unfreeze (55 layers vs original 30)
- **Batch size 32** — balances GPU memory usage and gradient stability

In [ ]:
# ── Configuration ──────────────────────────────────────────
IMG_SIZE    = (224, 224)    # ResNet50 standard resolution
BATCH_SIZE  = 32
NUM_CLASSES = 4

EPOCHS_P1, LR_P1 = 20, 1e-3      # Phase 1: head only (frozen backbone)
EPOCHS_P2, LR_P2 = 35, 1e-4      # Phase 2: top 55 ResNet layers — more epochs for deeper unfreeze
EPOCHS_P3, LR_P3 = 20, 5e-6      # Phase 3: all layers

# Validation split: 15 % of training data → ~950 images (was only 38 = too noisy)
VAL_SPLIT = 0.15

TRAIN_DIR = os.path.join(data_dir, 'train')
TEST_DIR  = os.path.join(data_dir, 'test')

print("✅ Configuration set")
print(f"   Backbone      : ResNet50  (224×224)")
print(f"   Batch         : {BATCH_SIZE}")
print(f"   Val split     : {int(VAL_SPLIT*100)} % of training data  (~stable graphs)")
print(f"   Phase 1       : {EPOCHS_P1} epochs  LR={LR_P1}")
print(f"   Phase 2       : {EPOCHS_P2} epochs  LR={LR_P2}  (top 55 layers unfrozen)")
print(f"   Phase 3       : {EPOCHS_P3} epochs  LR={LR_P3}")

✅ Configuration set
   Backbone      : ResNet50  (224×224)
   Batch         : 32
   Val split     : 15 % of training data  (~stable graphs)
   Phase 1       : 20 epochs  LR=0.001
   Phase 2       : 35 epochs  LR=0.0001  (top 55 layers unfrozen)
   Phase 3       : 20 epochs  LR=5e-06


### 2.4 Preprocessing Pipeline

#### CLAHE (Contrast Limited Adaptive Histogram Equalisation)
CLAHE is applied to every image before feeding it into the model. It enhances local contrast in X-ray images by operating in the **LAB colour space** on the luminance (L) channel only.

**Why CLAHE for X-rays?**
- Raw X-ray images often have low contrast in specific regions
- CLAHE enhances subtle features like lung opacities and infiltrates
- A standard contrast-enhancement step in chest X-ray pipelines
- `clipLimit=2.0` prevents over-amplification of noise
- `tileGridSize=(8,8)` ensures local (not global) contrast enhancement

#### Data Augmentation Strategy
Augmentation is applied **only to training images** to improve generalisation and reduce overfitting:

| Augmentation | Value | Reason |
|---|---|---|
| Rotation | ±15° | Slight patient positioning variation |
| Width/Height shift | 10% | Simulates different X-ray centering |
| Zoom | 10% | Scale variation in X-ray acquisition |
| Shear | 8% | Minor angular distortion |
| Brightness | 0.85–1.15 | Exposure variation between machines |
| **Horizontal flip** | **False ✗** | **Never flipped — heart position is diagnostically critical** |

> **Important:** Horizontal flip is explicitly disabled. The heart is anatomically on the left side of the chest. Flipping an X-ray moves the heart to the right, which would corrupt predictions. In our baseline model, TTA with horizontal flip caused accuracy to drop from 92.61% → 91.83%.

#### Validation Split
15% of the training folder is reserved as validation using `subset='validation'`. This gives ~948 validation images — compared to the original 38 — producing smooth, reliable training curves.

> **Note.** The validation subset is created from `train_datagen`, so validation images also receive random augmentation. Validation scores therefore vary slightly between passes (best 98.42 % during training vs 97.89 % when re-evaluated in Section 3.2). A separate, non-augmenting generator for the validation subset would remove this noise.

In [ ]:
# ── CLAHE + ResNet50 preprocessing ─────────────────────────
def clahe_preprocess(img):
    """
    Apply CLAHE to boost local contrast in X-ray images, then run
    the standard ResNet50 preprocessing (zero-centre by ImageNet mean).
    CLAHE is a common contrast-enhancement step for chest X-rays.
    """
    img    = np.array(img, dtype=np.uint8)
    lab    = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enh  = clahe.apply(l)
    merged = cv2.merge([l_enh, a, b])
    rgb    = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)
    return preprocess_input(rgb.astype(np.float32))

# ── Data generators ─────────────────────────────────────────
# NOTE: horizontal_flip=False for training AND TTA.
#       Chest X-rays must NEVER be flipped — the heart position (left side)
#       is a key diagnostic feature.  Flipping corrupts predictions.
train_datagen = ImageDataGenerator(
    preprocessing_function=clahe_preprocess,
    validation_split=VAL_SPLIT,   # 15 % of train folder used as validation
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=False,        # ← FIXED: never flip X-rays
    shear_range=0.08,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest'
)
val_test_datagen = ImageDataGenerator(
    preprocessing_function=clahe_preprocess   # CLAHE only, no augmentation
)

# Training subset (85 %)
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True, seed=42,
    subset='training')

# Validation subset (15 %) — same folder, different split
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False, seed=42,
    subset='validation')

# Test set — completely held-out, never used during training
test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False)

print("\n✅ Data generators ready")
print(f"   Class indices : {train_generator.class_indices}")
print(f"   Train images  : {train_generator.n}  ({100-int(VAL_SPLIT*100)} % of train folder)")
print(f"   Val images    : {val_generator.n}   ({int(VAL_SPLIT*100)} % of train folder  ← stable graphs)")
print(f"   Test images   : {test_generator.n}  (held-out)")
print(f"   Train batches : {len(train_generator)}")
print(f"   Val batches   : {len(val_generator)}")
print(f"   Test batches  : {len(test_generator)}")

Found 5378 images belonging to 4 classes.
Found 948 images belonging to 4 classes.
Found 771 images belonging to 4 classes.

✅ Data generators ready
   Class indices : {'COVID19': 0, 'NORMAL': 1, 'PNEUMONIA': 2, 'TURBERCULOSIS': 3}
   Train images  : 5378  (85 % of train folder)
   Val images    : 948   (15 % of train folder  ← stable graphs)
   Test images   : 771  (held-out)
   Train batches : 169
   Val batches   : 30
   Test batches  : 25


### 2.5 Handling Class Imbalance — Class Weights

The dataset is heavily imbalanced (Pneumonia: 3875, COVID-19: 460). We compute **balanced class weights** to penalise the model more heavily for misclassifying minority classes during training. Without this, the model would be biased toward predicting Pneumonia.

In [ ]:
# ── Class weights ───────────────────────────────────────────
train_labels        = train_generator.classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = dict(enumerate(class_weights_array))
idx_to_cls    = {v: k for k, v in train_generator.class_indices.items()}

print("✅ Class weights computed:")
for idx, w in class_weights.items():
    print(f"   {idx_to_cls[idx]:15s} → {w:.4f}")


✅ Class weights computed:
   COVID19         → 3.4386
   NORMAL          → 1.1794
   PNEUMONIA       → 0.4082
   TURBERCULOSIS   → 2.4313


### 2.6 ResNet50 Model Architecture

The base ResNet50 (frozen initially) is topped with a custom classification head:
```
ResNet50 (backbone)
    └── GlobalAveragePooling2D
    └── BatchNormalization
    └── Dense(512, relu) → Dropout(0.4)
    └── BatchNormalization
    └── Dense(256, relu) → Dropout(0.3)
    └── Dense(128, relu) → Dropout(0.2)
    └── Dense(4, softmax)
```
**Label smoothing = 0.1** is applied to the loss function to prevent the model from becoming over-confident on training data.

In [ ]:
# ── Build ResNet50 model with deeper head ───────────────────
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False    # frozen for Phase 1

# Deeper classification head  (512 → 256 → 128 → 4)
x      = base_model.output
x      = GlobalAveragePooling2D()(x)
x      = BatchNormalization()(x)
x      = Dense(512, activation='relu')(x)
x      = Dropout(0.4)(x)
x      = BatchNormalization()(x)
x      = Dense(256, activation='relu')(x)
x      = Dropout(0.3)(x)
x      = Dense(128, activation='relu')(x)
x      = Dropout(0.2)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

model   = Model(inputs=base_model.input, outputs=output)

# Label smoothing prevents over-confidence and improves generalisation
loss_fn = CategoricalCrossentropy(label_smoothing=0.1)

print("✅ ResNet50 model built  (deeper head)")
print(f"   Total layers         : {len(model.layers)}")
print(f"   Trainable layers     : {len([l for l in model.layers if l.trainable])}")
print(f"   Non-trainable layers : {len([l for l in model.layers if not l.trainable])}")
model.summary()


I0000 00:00:1773722567.708903      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773722567.715268      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ ResNet50 model built  (deeper head)
   Total layers         : 185
   Trainable layers     : 10
   Non-trainable layers : 175


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,811,780 (94.65 MB)

 Trainable params: 1,218,948 (4.65 MB)

 Non-trainable params: 23,592,832 (90.00 MB)

---

[← Dataset Understanding & Exploration](01_dataset_exploration.ipynb) · [Next: Three-Phase Fine-Tuning →](03_three_phase_fine_tuning.ipynb)